# Best-of-Public Portfolio — Verifiable Quantum Circuit Compilation (24.175x class)

This notebook **assembles and attributes** circuits for the three benchmarks into `submission.zip`. Every component is published by the teams credited below and was **re-verified locally with the official verifiers**.

For the chemistry task this portfolio goes one step further: we **re-derived the public ANGLE family's construction, reproduced part of it byte-for-byte, and then improved it** — four of the fifteen angle loaders in this submission are our own circuits (built with the public Apache-2.0 toolchain plus a kernel-relation search of our own), and the family scores strictly better than the best public chemistry circuit.

## Components, costs and attribution

| task | component | official cost | ratio vs baseline | provenance / license |
|---|---|---|---|---|
| invsqrt (fusion) | 62-qubit circuit, sha256 `bce2948f…fe47` | **100,568.4991945597** | 20.6503× | `yoyobar`'s 62-qubit circuit (`67482db6…`) with **Ehan Mallick's two 3→2 AND merges** (Apache-2.0; notebook `ehanmallick/invsqrt-20-65-two-3-to-2-and-merges-on-y6225`). The patch is public and rebuilds the circuit byte-for-byte — we reproduced sha256 `bce2948f…` on our own runner, and proved with an exact truth-table equivalence checker that no further 3→2 merge exists in this circuit. |
| dqi (combinatorial optimization) | 892-qubit decoder, sha256 `148bc9bc…f69b` | **377,371,305,874.26** | 64.7626× | `somejoint` / `yoyobar` fused confluent DQI family (v14). The decoder lineage descends from `jday96314`'s work (no published license) — credited here explicitly. The bytes are public in `yoyobar/qcc-local-sota-20261031`. |
| dfthcsos (chemistry) | 15 ANGLE circuits + factorization, submission sha256 `f6045aec…` | **317,614,412.40** (λ_eff 47.0302141, T 15,779, Q 428) | 10.5637× | **Lera** = `marculera`'s (9,143,40) factorization (Apache-2.0) with **heliosli's** table weights; the angle loaders follow `manwithacat`'s Hedge Wizardry block-design relation plans, built with the public compiler in `marculera/dfthc-compact-pipeline` (`dfthc_refit/tensorqrom.py`) and `lightningv08`'s kernel-relation tooling. **Loaders `angle_k_0..3` are our own**: we decoded the public construction, reproduced it byte-for-byte, then extended the kernel-relation set of each of those four loaders (75 → 77 relations, 314 → 313 Toffolis each) with a vertex-balanced pairing engine of our own. |

## Local re-verification with the official verifiers

- **invsqrt** — PASS: 100,568.4991945597 over the full domain (3,133,452 inputs × 64 shots = 200,540,928 shots), worst error/tolerance 0.9993237694056958 at x=161.
- **dqi** — PASS: 377,371,305,874.26 over 102,400 shots, 0 value errors, 0 phase errors, Q = 892, T_mean = 398,136.91.
- **dfthcsos** — PASS: 317,614,412.40 over 87,232 shots; λ_eff 47.0302141; E_gap 3.9148987010012206 (> 3.914898); Λ 284.4477; error_2 0.01999999978 (≤ 0.020); every ANGLE circuit ≤ 128 qubits (128 exactly).

## Reproducibility notes

- **invsqrt**: take the public base binary (sha256 `67482db6…`) and apply the public Apache-2.0 patch from `ehanmallick/invsqrt-20-65-two-3-to-2-and-merges-on-y6225`; the result is sha256 `bce2948f…` byte-for-byte.
- **dqi**: the decoder bytes are published in `yoyobar/qcc-local-sota-20261031` (sha256 `148bc9bc…`); the surrounding source lineage is credited in the table above.
- **dfthcsos**: the released circuits, the factorization and the table weights are all public (`heliosli/qcc-circuits-live`, `marculera/dfthc-compact-pipeline`, Apache-2.0). The per-task cost identity is `T_total = 17,286 − Σrel` with `λ_eff` and `Q_total` pinned by the factorization and the 128-qubit cap (selection 11 + J·brot 8 + cache 77 ⇒ J = 5), so the only free parameter of the chemistry circuit is the number of GF(2) kernel relations per loader. **Our four loaders are reproducible from the same public toolchain** by using our extended kernel-relation sets; the remaining eleven loaders are the public circuits, unmodified.

## Why a portfolio

The leaderboard score is the geometric mean of the three per-task improvement ratios, so the three components were chosen independently — each is the strongest circuit available for its own task. All are credited to their authors above and are used under their stated licenses (Apache-2.0 where the author has stated it).

## Our own measurements on these components (so the analysis can be reproduced)

- We re-derived the chemistry cost model from the official scorer and checked it against the official verifier: `chem = λ_eff × T_total × Q_total`, with the 128-qubit cap fixing `J = 5`, the high-cache decomposition fixing the 78 composite groups, the heuristic block fixed at 6,741 Toffolis, and the kernel-relation count as the only remaining degree of freedom.
- We rebuilt the SomeJoint q26 decoder (the byte-reproducible sibling of the DQI component above) from its 290 embedded source files plus 67 publicly downloadable inputs and obtained identical bytes — that rebuild is our deliverability fallback.
- For the fusion task we enumerated all 1,152 candidate AND merges of the public circuit with an exact truth-table equivalence checker (2,048 points per candidate): only the two merges already applied are legal, which is why that component cannot be improved further without a new construction.

In [ ]:

import zipfile, os, hashlib, sys

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

ROOT = '/kaggle/input'
wanted = {'invsqrt.kmb', 'dqi.kmb', 'submission.npz'}

def find_root():
    for dirpath, dirnames, filenames in os.walk(ROOT, followlinks=True):
        dirnames[:] = [d for d in dirnames if not d.startswith('.')]
        if wanted.issubset(set(filenames)) and any(f.startswith('angle_k_') for f in filenames):
            return dirpath
    return None

SRC = find_root()
if SRC is None:
    for dirpath, dirnames, filenames in os.walk(ROOT, followlinks=True):
        dirnames[:] = [d for d in dirnames if not d.startswith('.')]
        print(dirpath, '->', sorted(filenames)[:6], 'dirs:', dirnames[:6], file=sys.stderr)
    raise AssertionError('dataset not mounted under ' + ROOT)
print('dataset root:', SRC)

src_files = ['invsqrt.kmb', 'dqi.kmb'] + [f'angle_k_{i}.kmb' for i in range(15)] + ['submission.npz']
missing = [m for m in src_files if not os.path.exists(os.path.join(SRC, m))]
assert not missing, f'missing dataset members: {missing} (root {SRC})'

OUT = '/kaggle/working/submission.zip'
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_STORED) as z:
    z.write(os.path.join(SRC, 'invsqrt.kmb'), 'invsqrt.kmb')
    z.write(os.path.join(SRC, 'dqi.kmb'), 'dqi.kmb')
    for i in range(15):
        z.write(os.path.join(SRC, f'angle_k_{i}.kmb'), f'dfthcsos/angle_k_{i}.kmb')
    z.write(os.path.join(SRC, 'submission.npz'), 'dfthcsos/submission.npz')

with zipfile.ZipFile(OUT) as z:
    names = z.namelist()
    assert len(names) == 18, names
    assert 'invsqrt.kmb' in names and 'dqi.kmb' in names and 'dfthcsos/submission.npz' in names
    idx = sorted(int(n.split('_')[-1].split('.')[0]) for n in names if n.startswith('dfthcsos/angle_k_'))
    assert idx == list(range(15)), idx

for m in src_files:
    print(f'{m:22s} {sha256(os.path.join(SRC, m))[:16]}')
print('submission.zip ready:', os.path.getsize(OUT), 'bytes,', len(names), 'entries')
